[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/weilinear/rsi_labs/blob/main/gemini_self_improvement.ipynb)

# A bill-splitting agent that improves itself

One user prefers **tips on the subtotal before tax**. We will correct one answer,
retain that preference, have Gemini write a calculator and a simpler wrapper,
then revise the procedure that chooses repairs. The model stays fixed.

The first three improvements are directed demonstrations. The final experiment
asks the improver to choose interventions itself. Its score measures repair
selection, not end-to-end task success or open-ended recursive acceleration.

## Setup
Add `GOOGLE_API_KEY` to Colab **Secrets**, enable notebook access, and run the next
two cells. The secret is never printed or stored. Live calls consume API quota;
synthetic bills, feedback, and generated code are sent to Google. For local
Jupyter, install the SDK and skip the Colab secret cell with the key already in
the environment. No files or companion Python library are required.


In [ ]:
%pip -q install google-genai==2.28.0


In [ ]:
import os
from google.colab import userdata
os.environ["GOOGLE_API_KEY"] = userdata.get('GOOGLE_API_KEY')


## A few helpers
`ask()` uses the official SDK and JSON output. `run_tool()` accepts only a small
arithmetic subset of Python: one function, assignments, arithmetic, conditionals,
and approved calls. It checks generated functions before execution. This narrow
check is for this toy, **not a general sandbox for model-generated code**.

An independent Decimal calculation checks money results. Requests have a 45-second
timeout, no retries, and a 24-call cap. The two setup cells are not included in
saved execution counts; the experiment cells were verified with Python.


In [1]:
import ast, json, time, os
from decimal import Decimal, ROUND_HALF_UP
from google import genai
from google.genai import types

MODEL = os.environ.get("GEMINI_MODEL", "gemini-3.8-flash")
client = genai.Client(api_key=os.environ["GOOGLE_API_KEY"], vertexai=False,
    http_options=types.HttpOptions(timeout=45000,
        retry_options=types.HttpRetryOptions(attempts=1)))
usage = []

def ask(prompt, properties):
    if len(usage) >= 24:
        raise RuntimeError("Call budget exhausted")
    started = time.monotonic()
    response = client.models.generate_content(model=MODEL, contents=prompt,
        config={"temperature": 0, "max_output_tokens": 8192,
            "response_mime_type": "application/json",
            "response_json_schema": {"type": "object", "properties": properties,
                "required": list(properties), "additionalProperties": False},
            "automatic_function_calling": {"disable": True}})
    if not response.candidates or response.candidates[0].finish_reason != "STOP":
        raise RuntimeError("No complete model response: " + str(response.candidates[0].finish_reason if response.candidates else "no candidate"))
    usage.append({"tokens": response.usage_metadata.total_token_count,
                  "seconds": round(time.monotonic() - started, 2)})
    return json.loads(response.text)

def expected(subtotal, tax, tip, diners, tip_on="subtotal"):
    subtotal, tax, tip = map(lambda x: Decimal(str(x)), (subtotal, tax, tip))
    taxed = subtotal * (1 + tax / 100)
    basis = subtotal if tip_on == "subtotal" else taxed
    return ((taxed + basis * tip / 100) / diners).quantize(
        Decimal("0.01"), rounding=ROUND_HALF_UP)

def run_tool(source, name, arguments, extra=None):
    tree = ast.parse(source)
    allowed = (ast.Module, ast.FunctionDef, ast.arguments, ast.arg, ast.Assign,
        ast.Return, ast.Expr, ast.Name, ast.Load, ast.Store, ast.Constant,
        ast.BinOp, ast.Add, ast.Sub, ast.Mult, ast.Div, ast.IfExp, ast.Compare,
        ast.Eq, ast.Call, ast.Attribute, ast.keyword)
    nodes = list(ast.walk(tree))
    if len(source) > 2500 or len(nodes) > 220 or len(tree.body) != 1:
        raise ValueError("Tool exceeds the small arithmetic grammar")
    fn = tree.body[0]
    if not isinstance(fn, ast.FunctionDef) or fn.name != name or fn.decorator_list:
        raise ValueError("Expected exactly the requested function")
    if fn.args.defaults or fn.args.kw_defaults or fn.args.vararg or fn.args.kwarg:
        raise ValueError("No defaults or variadic arguments")
    approved = {"Decimal", "int", "str"} | set(extra or {})
    for node in nodes:
        if not isinstance(node, allowed):
            raise ValueError("Disallowed Python syntax: " + type(node).__name__)
        if isinstance(node, (ast.Name, ast.arg)) and "__" in getattr(node, "id", getattr(node, "arg", "")):
            raise ValueError("Disallowed name")
        if isinstance(node, ast.Attribute) and node.attr != "quantize":
            raise ValueError("Only Decimal.quantize is allowed")
        if isinstance(node, ast.Call):
            if isinstance(node.func, ast.Name) and node.func.id not in approved:
                raise ValueError("Disallowed function call")
            if not isinstance(node.func, (ast.Name, ast.Attribute)):
                raise ValueError("Disallowed call target")
        if isinstance(node, ast.Constant) and len(str(node.value)) > 100:
            raise ValueError("Constant too large")
    scope = {"__builtins__": {}, "Decimal": Decimal, "int": int, "str": str,
             "ROUND_HALF_UP": ROUND_HALF_UP, **(extra or {})}
    exec(compile(tree, "generated_arithmetic_tool", "exec"), scope)
    return scope[name](*arguments), scope[name]

policy = {"tip_on": "total"}  # A provisional convention, not a universal rule.
bill = {"subtotal": "100", "tax": "10", "tip": "20", "diners": 2}
print("Model:", MODEL)


Model: gemini-3.8-flash


## 1. Correct one answer, then retain a preference
The provisional policy tips on the taxed total: **$66 per person**. The user says,
“For my bills, calculate tips on the subtotal before tax.”

We first apply Gemini's proposed convention temporarily: **$65**. The unchanged
agent still returns $66 on its next request. Then we retain the convention in
`policy` and check a different bill. The information came from the user; the
agent did not discover a universal tipping rule.


In [2]:
feedback = "For my bills, calculate tips on the subtotal before tax."
proposal = ask(f"Current policy: {policy}. User feedback: {feedback}. "
               "Propose the user's tip convention.",
               {"tip_on": {"type": "string", "enum": ["subtotal", "total"]}})
print("Original:", expected(**bill, tip_on=policy["tip_on"]))
print("Temporary correction:", expected(**bill, tip_on=proposal["tip_on"]))
print("Next request without retention:", expected(**bill, tip_on=policy["tip_on"]))
# Retain only the user-supported convention.
if proposal["tip_on"] == "subtotal":
    policy = proposal
new_bill = {"subtotal": "137.45", "tax": "8.875", "tip": "18.5", "diners": 3}
print("Retained policy:", policy)
print("New bill per person:", expected(**new_bill, tip_on=policy["tip_on"]))


Original: 66.00
Temporary correction: 65.00
Next request without retention: 66.00
Retained policy: {'tip_on': 'subtotal'}
New bill per person: 58.36


## 2. Write a calculator
Ask Gemini to calculate the new bill without a tool; report whether it actually
makes an error. A correct answer is allowed. We then ask it to write a reusable
calculator, so money arithmetic can be executed and checked rather than entrusted
to generated text.

The first tool uses an awkward legacy contract: **cents** and **basis points**
(100 basis points = 1%). We deliberately keep that interface for the next section.
The generated tool is accepted only if it passes all three development checks;
a fourth bill is checked after acceptance.


In [3]:
manual = ask(f"Calculate the amount per person for {new_bill}; tip_on={policy['tip_on']}. "
             "Do the arithmetic yourself and return dollars as a decimal string.",
             {"amount": {"type": "string"}})
print("Model without calculator:", manual["amount"],
      "| Correct:", Decimal(manual["amount"]) == expected(**new_bill))
calculator_code = ask("Write one Python function calculator(subtotal_cents, tax_basis_points, "
    "tip_basis_points, tip_basis, diners). Return a Decimal dollar amount per person, "
    "rounded to cents with ROUND_HALF_UP. tip_basis is 'subtotal' or 'total'. "
    "Inputs are numeric cents and basis points; basis points may be fractional. Use Decimal(str(...)) for values. "
    "Allowed: assignments, arithmetic, a conditional expression, Decimal.quantize. "
    "Decimal, str, int, ROUND_HALF_UP are provided. No imports, defaults, loops, "
    "annotations, markdown fences, or other functions.", {"code": {"type": "string"}})["code"]
checks = [(10000, 1000, 2000, "subtotal", 2),
          (10000, 1000, 2000, "total", 2), (13745, 887.5, 1850, "subtotal", 3)]
calculator = None
for args in checks:
    result, candidate = run_tool(calculator_code, "calculator", args)
    oracle = expected(Decimal(str(args[0])) / 100, Decimal(str(args[1])) / 100,
                      Decimal(str(args[2])) / 100, args[4], args[3])
    assert result == oracle, "Generated calculator failed; do not retain it"
calculator = candidate
print(calculator_code)
held_out = (8643, 725, 1750, "subtotal", 4)
assert calculator(*held_out) == expected("86.43", "7.25", "17.5", 4)
print("Retained calculator: three development checks and one held-out check passed.")


Model without calculator: 58.36 | Correct: True
def calculator(subtotal_cents, tax_basis_points, tip_basis_points, tip_basis, diners):
    subtotal = Decimal(str(subtotal_cents)) / Decimal('100')
    tax = subtotal * Decimal(str(tax_basis_points)) / Decimal('10000')
    tip_base = subtotal + tax if tip_basis == 'total' else subtotal
    tip = tip_base * Decimal(str(tip_basis_points)) / Decimal('10000')
    return ((subtotal + tax + tip) / Decimal(str(diners))).quantize(Decimal('0.01'), rounding=ROUND_HALF_UP)
Retained calculator: three development checks and one held-out check passed.


## 3. Write a simpler wrapper
Here is a **constructed regression**, not a claimed Gemini mistake: the agent
passes dollars and percentages directly to a function expecting cents and basis
points. The calculator runs correctly on the wrong units and produces the wrong
bill. Another reminder in the prompt might help; a wrapper can remove conversion
work from the caller.

Gemini writes `split_bill(subtotal_dollars, tax_percent, tip_percent, tip_on, diners)`.
We retain it only after checking three bills. It converts units and delegates to
the existing calculator; it does not duplicate the arithmetic.


In [4]:
wrong_units = calculator(100, 10, 20, policy["tip_on"], 2)
print("Constructed wrong-unit call:", wrong_units, "| Expected:", expected(**bill))
wrapper_code = ask("Write one function split_bill(subtotal_dollars, tax_percent, "
    "tip_percent, tip_on, diners). Delegate to the provided calculator with positional "
    "arguments (subtotal_cents, tax_basis_points, tip_basis_points, tip_basis, diners). "
    "Convert dollars to cents and percent to basis points, preserving fractional "
    "basis points. Provided: calculator, Decimal, str, int. Return its Decimal result. "
    "Only assignments, arithmetic and approved calls. No imports, defaults, loops, "
    "annotations or markdown fences.", {"code": {"type": "string"}})["code"]
wrapper_code = ast.unparse(ast.parse(wrapper_code))
# Contract is positional; wrapper caller supplies human-facing units.
for task in (bill, new_bill, {"subtotal": "86.43", "tax": "7.25", "tip": "17.5", "diners": 4}):
    args = (task["subtotal"], task["tax"], task["tip"], policy["tip_on"], task["diners"])
    result, candidate = run_tool(wrapper_code, "split_bill", args, {"calculator": calculator})
    assert result == expected(**task), "Generated wrapper failed; do not retain it"
split_bill = candidate
print(wrapper_code)
print("Human-facing call:", split_bill("100", "10", "20", policy["tip_on"], 2))


Constructed wrong-unit call: 0.50 | Expected: 65.00
def split_bill(subtotal_dollars, tax_percent, tip_percent, tip_on, diners):
    subtotal_cents = int(Decimal(str(subtotal_dollars)) * 100)
    tax_basis_points = Decimal(str(tax_percent)) * 100
    tip_basis_points = Decimal(str(tip_percent)) * 100
    tip_basis = tip_on
    return calculator(subtotal_cents, tax_basis_points, tip_basis_points, tip_basis, diners)
Human-facing call: 65.00


## 4. Improve how the agent chooses improvements
Our initial improver responds to **every** failure with another instruction.
That works for a missing user convention, but fails to distinguish arithmetic
execution from an awkward tool interface.

We score repair selection on three labeled development failure cases, ask Gemini
to revise the improver's instructions, and accept only a higher-scoring candidate.
The improver can revise its own instruction artifact under its original
instruction-editing policy. Its replacement then governs future repair selection.

The three held-out cases use different bills and descriptions. Labels and failures
are human-authored fixtures, and the score is **repair-choice accuracy**. This
small proxy does not prove that every selected repair will improve a real agent.


In [5]:
ACTIONS = ["update_instruction", "write_calculator", "write_wrapper"]
repair_schema = {"action": {"type": "string", "enum": ACTIONS}}
improver = "For every failure, recommend update_instruction. Do not recommend writing tools."
development = [
    ("User says: I tip on the subtotal, but the agent tips on the taxed total.", "update_instruction"),
    ("No arithmetic tool exists. The formula is right, but a manual sum differs from the reference.", "write_calculator"),
    ("A tested calculator exists. The caller passed $100 as 100 cents and 20% as 20 basis points.", "write_wrapper"),
]
held_out_cases = [
    ("This user's $86.43 meal was tipped after tax; they explicitly require tipping before tax.", "update_instruction"),
    ("No calculator is available. A correct multi-step bill formula produced an incorrect decimal result.", "write_calculator"),
    ("The calculator passed its tests. A caller supplied 7.25 for a parameter expecting 725 basis points.", "write_wrapper"),
]

def choose_repair(instruction, case):
    return ask(f"Improvement policy: {instruction}\nFailure evidence: {case}\n"
               "Choose the repair according to the current improvement policy.", repair_schema)["action"]

def score_improver(instruction, cases):
    predictions = [choose_repair(instruction, evidence) for evidence, _ in cases]
    score = sum(pred == label for pred, (_, label) in zip(predictions, cases)) / len(cases)
    return score, predictions

before, predictions = score_improver(improver, development)
meta_feedback = [{"evidence": evidence, "chosen": pred, "expected": label}
                 for (evidence, label), pred in zip(development, predictions)]
meta_change = ask(f"You are the improver. Your current policy is: {improver}. "
    "Apply its instruction-editing procedure to YOUR OWN improvement instructions. "
    f"Feedback on repair choices: {json.dumps(meta_feedback)}. "
    "Propose a replacement policy that makes future repair choices more accurate. "
    "Keep it under 100 words.", {"instruction": {"type": "string", "maxLength": 800}})["instruction"]
after, _ = score_improver(meta_change, development)
if after > before:
    improver = meta_change  # Retain a measured improvement, not merely a rewrite.
old_held_out, _ = score_improver(
    "For every failure, recommend update_instruction. Do not recommend writing tools.", held_out_cases)
new_held_out, _ = score_improver(improver, held_out_cases)
print("Retained improvement policy:", improver)
print("Development repair-choice accuracy:", before, "→", after)
print("Held-out repair-choice accuracy:", old_held_out, "→", new_held_out)
print("Model calls:", len(usage), "| Reported tokens:", sum(c['tokens'] or 0 for c in usage))


Retained improvement policy: Evaluate failure root causes systematically: recommend update_instruction when the failure stems from missing domain guidelines, behavioral rules, or intent ambiguity. Recommend writing specialized tools (such as write_calculator) when precise computation, external capability, or algorithmic execution is absent. Recommend write_wrapper when an appropriate tool exists but input/output units, formats, or interface contracts require conversion.
Development repair-choice accuracy: 0.3333333333333333 → 1.0
Held-out repair-choice accuracy: 0.3333333333333333 → 1.0
Model calls: 17 | Reported tokens: 16263


## What changed?

| Change | Editable object | Retained? |
|---|---|---|
| Correct this bill | Temporary convention | No |
| Learn the user's convention | Task policy | Yes |
| Create reliable arithmetic | Calculator code | Yes, after checks |
| Simplify a tool interface | Wrapper code | Yes, after checks |
| Learn which repair to propose | Improvement policy | Yes, if its score improves |

The objective, permitted syntax, model, and evaluator remain fixed. Coding expands
the editable machinery; recursion comes from changing the procedure that generates
future improvements. There is no claim of unlimited acceleration or equal compute.
The tiny diagnostics test needs repeated runs and harder held-out cases before
we could claim a broadly better improver. Nothing is published from notebook runs;
retained artifacts live in the current notebook session.
